# Driver or car? A causal attribution that fails *silently*, and how to fix it

*How much of a Formula 1 result is the **driver**, and how much is the **car**?* It's the
sport's eternal bar argument, and it's a textbook causal question: `do(car = X)` holding the
driver fixed. This notebook answers it with DoWhy's `gcm` module, but that's not why it exists.

**It exists because the first attempt fails, and fails silently.** Every API call succeeds. No
warnings. The diagnostics look fine. And the answer is confidently backwards. The failure isn't in
the library: it's a *structural non-identifiability* in how the question was encoded, and no
amount of fitting can overcome it. Notably, it is a failure that DAG falsification does **not**
catch: the graph is right; the variable encoding is what's broken.

The arc:

1. **The naive model:** categorical driver/car nodes; everything runs; the answer is wrong.
2. **The diagnosis:** one number (Cramér's V) explains why it *had* to be wrong.
3. **The fix:** identification happens *upstream*: continuous skill/pace latents from teammate
   comparisons, fed into the *same* graph with the same API calls.
4. **Honest caveats:** what moves the answer (era, a confounding edge) and which measures are
   robust to that.

Runs in under a minute on one CPU core with a fixed seed. The data (a single small CSV,
[f1db](https://github.com/f1db/f1db) CC-BY-4.0) and the precomputed latents come from
[apex-attribution](https://github.com/bolt-chaos/apex-attribution), where the full model is
built, validated out-of-sample, and documented.


## 0. Setup and data

Everything below runs on DoWhy's own dependency set (`poetry install -E plotting`).


In [ ]:
import dowhy
dowhy.enable_notebook_rendering()

import warnings
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

import dowhy.gcm as gcm
from dowhy.utils import plot

warnings.filterwarnings("ignore")  # sklearn fit chatter; nothing load-bearing
gcm.util.general.set_random_seed(20260816)


### Data

`f1_attribution_2018_2025.csv` (312 KB): one row per (driver, race), 2018 to 2025, the
41-driver teammate-connected cohort. It holds observed race data (grid, finish, circuit type,
driver and constructor ids) **plus two precomputed columns**, `driver_skill` and `car_pace`,
which are posterior means from a hierarchical Bayesian model identified by teammate contrasts
(skill follows a per-season random walk). The upstream model, its out-of-sample validation and
the full pipeline live in [apex-attribution](https://github.com/bolt-chaos/apex-attribution).

| column | meaning |
|---|---|
| `year`, `round`, `race_id` | season / race identifiers, 2018 to 2025 |
| `circuit_type` | categorical circuit class (`RACE`, `STREET`, `ROAD`, as in f1db) |
| `driver_id`, `constructor_id` | observed driver and team: the naive cause nodes |
| `team_year` | team-season key (`constructor@year`), used to build the teammate graph in §4 |
| `grid`, `finish_pos` | starting and finishing position (`finish_pos` is defined for classified finishes only, hence the `classified` flag) |
| `driver_skill`, `car_pace` | precomputed posterior means of the continuous latents (identified from teammate qualifying gaps; see [apex-attribution](https://github.com/bolt-chaos/apex-attribution)): the fixed cause nodes |

Raw data: [f1db](https://github.com/f1db/f1db), CC-BY-4.0; the CSV is a derived subset and
retains that attribution.


In [ ]:
import pandas as pd

full = pd.read_csv("f1_attribution_2018_2025.csv")
df = full[full.classified].copy()          # finish_pos is defined for classified finishes only
for c in ["grid", "finish_pos", "driver_skill", "car_pace"]:
    df[c] = df[c].astype(float)
for c in ["circuit_type", "driver_id", "constructor_id"]:
    df[c] = df[c].astype("object")

# the "current regulations" window a careful analyst would naturally pick (see §2)
df22 = df[df.year >= 2022].copy()

print(f"{len(df)} classified results, {df.driver_id.nunique()} drivers, "
      f"{df.constructor_id.nunique()} constructors, {df.year.min()}-{df.year.max()}"
      f"   (current-regs subset 2022-2025: {len(df22)} results)")
df[["year", "circuit_type", "driver_id", "constructor_id", "grid", "finish_pos",
    "driver_skill", "car_pace"]].head(3)


## 1. The question, posed causally

Five variables tell the story of a race result: who is driving, what they're driving, the kind
of circuit, where they start, and where they finish. The graph is uncontroversial: driver and
car affect qualifying (grid) and the race itself; the circuit type moderates both; grid position
carries a real causal effect of its own (track position matters):

```
circuit_type ─┐
driver ───────┼──→ grid ────────┐
car ──────────┘                 ├──→ finish_pos
driver ─────────────────────────┤
car ────────────────────────────┤
circuit_type ───────────────────┘
```

The naive encoding uses the two *observed* categorical columns as the cause nodes:
`driver_id` (who) and `constructor_id` (which team's car).

In [ ]:
EDGES_NAIVE = [("circuit_type", "grid"), ("driver_id", "grid"), ("constructor_id", "grid"),
               ("circuit_type", "finish_pos"), ("driver_id", "finish_pos"),
               ("constructor_id", "finish_pos"), ("grid", "finish_pos")]
NODES_NAIVE = ["circuit_type", "driver_id", "constructor_id", "grid", "finish_pos"]

g_naive = nx.DiGraph(EDGES_NAIVE)
assert nx.is_directed_acyclic_graph(g_naive)
print("nodes:", list(g_naive.nodes))
print("edges:", g_naive.number_of_edges())

In [ ]:
plot(g_naive)


## 2. The naive model: every call succeeds

One more decision a careful analyst makes *before* modelling: **restrict to the current
regulation era.** Formula 1 rewrote its technical rules for 2022; mixing the old cars with the
new would confound everything, so we fit on **2022–2025**, the methodologically cautious
choice.

Then the standard `gcm` workflow: auto-assign mechanisms, fit, evaluate the fitted model, and
ask the attribution question two ways: the **intrinsic causal influence** variance
decomposition, and a direct **interventional comparison** (`do()` on the car with the driver
fixed, and vice versa).
Reduced sample counts keep this notebook fast; the direction of the result is stable across
counts (fixed seed).

In [ ]:
scm_naive = gcm.InvertibleStructuralCausalModel(g_naive)
gcm.auto.assign_causal_mechanisms(scm_naive, df22[NODES_NAIVE], quality=gcm.auto.AssignmentQuality.GOOD)
gcm.fit(scm_naive, df22[NODES_NAIVE])
print("fitted — no warnings, no errors")

In [ ]:
print(gcm.evaluate_causal_model(scm_naive, df22[NODES_NAIVE],
                                evaluate_causal_structure=False,
                                evaluate_invertibility_assumptions=False))


Read the report the way a careful analyst would: every root node reproduces its marginal
"very well", `grid` scores "fair" and `finish_pos` "good" on CRPS. The one caution is the joint
distribution: the model draws its roots independently, and drivers and teams are far from
independent in the data, so the generated joint mismatches the observed one. That is a warning
about the model of the *joint*, and the correctly specified model in §4 draws the same caution
(its roots are correlated too; see the correlation printed in §5). Nothing here points at the
encoding, and nothing says the question is unidentified.


First the variance decomposition (the following cell takes about 5 to 10 s). Confidence intervals are omitted here for docs-CI runtime; the full analysis reports 90% credible intervals (see §5 table).


In [ ]:
ICC_RAND_NAIVE, ICC_BASE_NAIVE = 20, 80   # reduced for runtime; direction is what matters

icc_naive = gcm.intrinsic_causal_influence(
    scm_naive, "finish_pos",
    num_samples_randomization=ICC_RAND_NAIVE, num_samples_baseline=ICC_BASE_NAIVE)

tot = sum(abs(v) for v in icc_naive.values())
share_naive = {k: abs(v) / tot for k, v in icc_naive.items()}
for k, v in sorted(share_naive.items(), key=lambda kv: -kv[1]):
    print(f"  {k:16s} {100*v:5.1f}%")
print(f"\n  driver_id {100*share_naive['driver_id']:.1f}%  vs  "
      f"constructor_id {100*share_naive['constructor_id']:.1f}%")

Then the interventional comparison: hold the driver fixed and swap the car across the field, then the reverse.


In [ ]:
def exp_finish_naive(driver, constructor, n=800):
    """E[finish | do(driver), do(constructor)] on 2022-2025 — circuit marginalized."""
    iv = {"driver_id": lambda x, d=driver: d, "constructor_id": lambda x, c=constructor: c}
    return float(gcm.interventional_samples(scm_naive, iv, num_samples_to_draw=n)["finish_pos"].mean())

# hold the DRIVER fixed, swap the CAR across the field's best/worst
swap_car = {c: exp_finish_naive("max-verstappen", c)
            for c in ["red-bull", "mercedes", "haas", "williams"]}
# hold the CAR fixed, swap the DRIVER
swap_drv = {d: exp_finish_naive(d, "red-bull")
            for d in ["max-verstappen", "lewis-hamilton", "lance-stroll", "nicholas-latifi"]}

car_effect_naive = max(swap_car.values()) - min(swap_car.values())
drv_effect_naive = max(swap_drv.values()) - min(swap_drv.values())
print("Verstappen, swept across cars :", {k: round(v, 1) for k, v in swap_car.items()})
print("Red Bull, swept across drivers:", {k: round(v, 1) for k, v in swap_drv.items()})
print(f"\n  car effect {car_effect_naive:.1f} positions  vs  driver effect {drv_effect_naive:.1f} positions")

**The answer is confidently backwards.** (Reading the ICC list: `finish_pos`'s own large share
is the outcome's *irreducible race-day noise*, i.e. luck; the attribution question is the comparison
between the two cause nodes.) The variance share puts nearly everything on the
driver and almost nothing on the car, and the interventional sweep agrees: swap Verstappen's
car from a Red Bull to a Williams and the model barely moves him; swap the driver inside a Red
Bull and the prediction swings by many positions.

Anyone who follows the sport knows this is wrong: under these regulations the *car* is the
dominant factor. That's the project's built-in sanity check, and the naive model fails it. But
notice what did **not** happen: no API call failed, and nothing in the evaluation report pointed
at the encoding. If you didn't have domain ground truth to check against, you would ship this
number.

## 3. The diagnosis: one number explains it

Why did the model *have* to fail? Because of who drives what. A driver almost never changes
teams mid-season, and rarely between seasons, so the two categorical columns we asked `gcm` to
separate are nearly the *same column under two names*. The standard measure of association
between two categoricals is **Cramér's V** (0 = independent, 1 = one determines the other):

In [ ]:
ct = pd.crosstab(df22.driver_id, df22.constructor_id)
chi2 = stats.chi2_contingency(ct)[0]
n = ct.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
print(f"Cramér's V (driver_id × constructor_id) = {cramers_v:.2f}")

That is categorical near-collinearity. With `do(constructor = williams)` for Verstappen, we're
asking about rows that essentially don't exist: the model has almost no data where the driver
column and the team column decouple, so it is free to load the joint driver+car effect onto
either variable. It happened to pick the finer-grained one (there are more drivers than teams,
so `driver_id` can absorb more variance). The fit is fine; the **question is not identified**
from this encoding.

Two things make this failure mode treacherous:

- **Falsification does not catch it.** `gcm.falsify_graph` tests whether the *graph* is
  consistent with the data's conditional independencies, and this graph is. The problem lives
  in the *variable encoding*, one level below anything a graph test can see. (We verified this
  on the full model in [apex-attribution](https://github.com/bolt-chaos/apex-attribution); it's
  skipped here for runtime.)
- **It's common.** Any panel where units rarely switch groups has it: employees nested in
  companies, students in schools, patients in hospitals. If you'd ask "how much is the person
  vs. the institution," this is your problem too.

## 4. The fix: identification happens *upstream* of the SCM

> *"No causes in, no causes out."*
>
> Nancy Cartwright, *Nature's Capacities and Their Measurement* (1989), ch. 2

The signal that separates driver from car **is in the data**, just not in the categorical
encoding. Teammates drive the *same car*, so within a team-season, the gap between two drivers'
qualifying times is car-free: the car cancels, leaving pure driver skill. And when drivers
switch teams across seasons, those within-team comparisons **chain into a connected graph**
across the whole grid, the same trick that makes chess Elo work across players who never met.

But chaining needs the graph to actually *connect*, and this is where the cautious
2022–2025 window quietly sabotaged us a second time:

In [ ]:
def teammate_components(rows):
    """Graph over drivers, an edge when two drivers shared a team_year -> component sizes."""
    g = nx.Graph()
    for _, grp in rows.groupby("team_year"):
        ds = grp.driver_id.unique()
        g.add_nodes_from(ds)
        g.add_edges_from((a, b) for i, a in enumerate(ds) for b in ds[i + 1:])
    return sorted((len(c) for c in nx.connected_components(g)), reverse=True)

print("teammate-graph connected components (drivers per island):")
print(f"  2022-2025 (narrow): {teammate_components(full[full.year >= 2022])}")
print(f"  2018-2025 (wide):   {teammate_components(full)}")

The narrow window fragments the grid into disconnected islands: a driver's skill can only be
compared *within* their island, so the scale between islands (and every backmarker stuck on a
small one) is unidentified. Reaching back to 2018 connects **every driver into one component**
(e.g. Albon alone links Red Bull ↔ Toro Rosso ↔ Williams). Identification here is a property of
the *data*, and you can check it directly: this one `networkx` call is the difference between
a defensible latent and a hallucinated one.

[apex-attribution](https://github.com/bolt-chaos/apex-attribution) exploits the connected wide
window with a hierarchical Bayesian model (driver skill and per-team-season car pace as latent
variables, identified by exactly those teammate contrasts, with skill following a random walk
across seasons). Fitting it takes PyMC and a few minutes, so this notebook ships the
**posterior means** as two precomputed columns, `driver_skill` and `car_pace` (units: % of
qualifying lap time vs. the field; lower = faster). The upstream model is validated
out-of-sample: fit on 2018–2023, it predicts held-out 2024–2025 teammate head-to-heads at 80%
season-long (coin-flip baseline 50%).

The SCM below is the **same five-node structure, the same causal query, the same API calls**;
only the categorical proxies are replaced by the continuous latents they were standing in for,
fit on the connected 2018–2025 window.

In [ ]:
NODES = ["circuit_type", "driver_skill", "car_pace", "grid", "finish_pos"]
EDGES = [("circuit_type", "grid"), ("driver_skill", "grid"), ("car_pace", "grid"),
         ("circuit_type", "finish_pos"), ("driver_skill", "finish_pos"),
         ("car_pace", "finish_pos"), ("grid", "finish_pos")]

g_fix = nx.DiGraph(EDGES)
plot(g_fix)


Same structure, same calls: assign mechanisms and fit, now on the connected 2018 to 2025 window.


In [ ]:
scm = gcm.InvertibleStructuralCausalModel(g_fix)
gcm.auto.assign_causal_mechanisms(scm, df[NODES], quality=gcm.auto.AssignmentQuality.GOOD)
gcm.fit(scm, df[NODES])


In [ ]:
ICC_RAND, ICC_BASE = 30, 120
icc_fix = gcm.intrinsic_causal_influence(scm, "finish_pos",
                                         num_samples_randomization=ICC_RAND,
                                         num_samples_baseline=ICC_BASE)
tot = sum(abs(v) for v in icc_fix.values())
share_fix = {k: abs(v) / tot for k, v in icc_fix.items()}
for k, v in sorted(share_fix.items(), key=lambda kv: -kv[1]):
    print(f"  {k:16s} {100*v:5.1f}%")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for ax, shares, title, keys in [
        (axes[0], share_naive, "naive: categoricals, 2022-2025",
         ["driver_id", "constructor_id", "grid", "circuit_type"]),
        (axes[1], share_fix, "fixed: latents, connected 2018-2025",
         ["driver_skill", "car_pace", "grid", "circuit_type"])]:
    labels = [k.replace("_id", "").replace("_", " ") for k in keys]
    vals = [100 * shares.get(k, 0) for k in keys]
    colors = ["#3a6ea5" if "driver" in k or "skill" in k
              else "#e10600" if "constructor" in k or "pace" in k else "#b9b9b9" for k in keys]
    ax.barh(labels[::-1], vals[::-1], color=colors[::-1])
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("share of finish-position variance (%)")
fig.suptitle("Same graph structure, same calls — the encoding (and its identifying window) changed", fontsize=11)
fig.tight_layout()
plt.show()

The variance share now behaves, but a share is a *descriptive* number (more on that in §5).
The measures to lead with are the ones that answer the actual question: **interventions** and
**counterfactuals**, in units anyone can read: finishing positions.

In [ ]:
def exp_finish(skill, pace, n=800):
    """E[finish | do(driver_skill=skill), do(car_pace=pace)] — circuit marginalized."""
    iv = {"driver_skill": lambda x, s=skill: s, "car_pace": lambda x, p=pace: p}
    return float(gcm.interventional_samples(scm, iv, num_samples_to_draw=n)["finish_pos"].mean())

skill_by_d = df.groupby("driver_id").driver_skill.mean()      # per-driver career skill
pace_by_ty = df.groupby("team_year").car_pace.first()          # per team-season car pace
pace_by_c = df.groupby("constructor_id").car_pace.mean()
mid_skill, mid_pace = float(skill_by_d.median()), float(pace_by_ty.median())


Sweep each factor over its observed range (best to worst driver; best to worst car), holding the other at the field median:


In [ ]:
# sweep each factor over its OBSERVED range (best driver -> worst driver; best car -> worst
# car), holding the other at the field median
pace_grid = np.linspace(pace_by_ty.min(), pace_by_ty.max(), 7)
skill_grid = np.linspace(skill_by_d.min(), skill_by_d.max(), 7)
car_curve = [exp_finish(mid_skill, p) for p in pace_grid]
drv_curve = [exp_finish(s, mid_pace) for s in skill_grid]
car_effect = max(car_curve) - min(car_curve)
drv_effect = max(drv_curve) - min(drv_curve)


Plot both curves in the same units, finishing positions, and compare with the naive numbers:


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(np.linspace(0, 1, 7), car_curve, "o-", color="#e10600",
        label=f"sweep the CAR (median driver): {car_effect:.1f} positions")
ax.plot(np.linspace(0, 1, 7), drv_curve, "o-", color="#3a6ea5",
        label=f"sweep the DRIVER (median car): {drv_effect:.1f} positions")
ax.set_xlabel("factor swept from best observed → worst observed")
ax.set_ylabel("expected finishing position")
ax.invert_yaxis(); ax.legend(fontsize=9)
ax.set_title("Interventional effect sizes, in positions (the graph-robust answer)")
fig.tight_layout(); plt.show()

print(f"car effect {car_effect:.1f} positions  vs  driver effect {drv_effect:.1f} positions"
      f"   (naive said {car_effect_naive:.1f} vs {drv_effect_naive:.1f})")


And the rung-3 party trick the naive model could never do: a **counterfactual** for one specific
race. Alex Albon finished P13 in a Williams. Same driver, same race-day luck (the abducted
noise), but *counterfactually* in a Red Bull?

In [ ]:
row = df[(df.driver_id == "alexander-albon") & (df.constructor_id == "williams")].iloc[[0]]
obs = row[NODES].copy()

cf = gcm.counterfactual_samples(
    scm, {"car_pace": lambda x, p=float(pace_by_c["red-bull"]): p}, observed_data=obs)

print(f"Albon, {int(row.year.iloc[0])}: finished P{int(row.finish_pos.iloc[0])} in the Williams")
print(f"same driver, same luck, Red Bull car -> P{cf.finish_pos.iloc[0]:.0f}")

## 5. Honest caveats: what moves this answer

> *"Behind every causal conclusion there must lie some causal assumption that is not
> testable in observational studies."*
>
> Judea Pearl, *Causal inference in statistics: An overview*, Statistics Surveys 3 (2009), §2.1

**The split is era-dependent.** "X% driver / Y% car" is a property of the *population in the
window*, not a law of the sport. Wider windows contain more car variation, so the car explains
more ([full analysis](https://github.com/bolt-chaos/apex-attribution)):

| era window | car share (median, 90% CrI) | driver share | P(car > driver) |
|---|---|---|---|
| 2018–2025 (regulation-converged) | 32% [23, 42] | 21% [13, 29] | 73% |
| 2006–2025 (20 years) | 44% [35, 48] | 12% [6, 15] | 100% |

**Was it the encoding or the window? Both, through one mechanism: decoupling variation.**
Widening the window adds team-switchers, which helps *any* encoding (the categorical model also
partially recovers on 2018–2025). What the latent route buys is that the identification is
**explicit and checkable**: the teammate-graph connectivity test above tells you *in advance*
whether the comparisons chain, instead of leaving you to notice a wrong answer. It also buys
quantities categories cannot express: "a car 1% faster," and the counterfactual you just ran.

**And the variance share is graph-dependent.** `driver_skill` and `car_pace` are correlated
(good drivers get hired into good cars); the check below prints the correlation. Modelling that
hiring pathway as an explicit edge (`driver_skill → car_pace`) instead of leaving the roots
independent changes *nothing* about the data or the fit quality, but watch what it does to the
two kinds of answer:

In [ ]:
print(f"corr(driver_skill, car_pace) = {df.driver_skill.corr(df.car_pace):.2f}\n")

scm_hire = gcm.InvertibleStructuralCausalModel(nx.DiGraph(EDGES + [("driver_skill", "car_pace")]))
gcm.auto.assign_causal_mechanisms(scm_hire, df[NODES], quality=gcm.auto.AssignmentQuality.GOOD)
gcm.fit(scm_hire, df[NODES])

icc_h = gcm.intrinsic_causal_influence(scm_hire, "finish_pos",
                                       num_samples_randomization=ICC_RAND,
                                       num_samples_baseline=ICC_BASE)
tot_h = sum(abs(v) for v in icc_h.values())
share_h = {k: abs(v) / tot_h for k, v in icc_h.items()}


Now the same interventional car sweep on the model with the hiring edge, side by side with the independent-roots model:


In [ ]:
def exp_finish_h(skill, pace, n=800):
    iv = {"driver_skill": lambda x, s=skill: s, "car_pace": lambda x, p=pace: p}
    return float(gcm.interventional_samples(scm_hire, iv, num_samples_to_draw=n)["finish_pos"].mean())

car_effect_h = (max(exp_finish_h(mid_skill, p) for p in [pace_grid[0], pace_grid[-1]])
                - min(exp_finish_h(mid_skill, p) for p in [pace_grid[0], pace_grid[-1]]))

print("                       independent roots   + hiring edge")
print(f"  ICC car share            {100*share_fix['car_pace']:5.1f}%          {100*share_h['car_pace']:5.1f}%")
print(f"  ICC driver share         {100*share_fix['driver_skill']:5.1f}%          {100*share_h['driver_skill']:5.1f}%")
print(f"  interventional car       {car_effect:5.1f} pos        {car_effect_h:5.1f} pos")


The ICC split **swings wildly on a modelling choice the data cannot arbitrate**, because
`intrinsic_causal_influence` decomposes variance under an independent-root-noise assumption
that the hiring edge violates by construction. The interventional effect barely moves, because
`do()` sets both roots and doesn't care how their observational correlation arose.

**The practical lesson:** when your root causes are correlated, treat the ICC share as
descriptive, and lead with interventional and counterfactual quantities. One more of those is
a *probability of necessity*: of the podiums actually achieved, what fraction would have been
lost **but for** the car (counterfactually downgraded to midfield) vs. **but for** the driver
(downgraded to the median)?

In [ ]:
podiums = df[df.finish_pos <= 3][NODES].copy()

cf_car = gcm.counterfactual_samples(
    scm, {"car_pace": lambda x: np.full(np.shape(x), mid_pace)}, observed_data=podiums)
cf_drv = gcm.counterfactual_samples(
    scm, {"driver_skill": lambda x: np.full(np.shape(x), mid_skill)}, observed_data=podiums)

pn_car = float((cf_car.finish_pos > 3).mean())
pn_drv = float((cf_drv.finish_pos > 3).mean())
print(f"of {len(podiums)} podiums achieved 2018-2025:")
print(f"  {100*pn_car:.0f}% are lost but-for the car (midfield machinery instead)")
print(f"  {100*pn_drv:.0f}% are lost but-for the driver (median driver instead)")
print("\n  -> most podiums needed BOTH; the car is necessary slightly more often")

## 6. Takeaways

1. **A causal pipeline can succeed at every step and still answer the wrong question.** Nothing
   in fit/evaluate/attribute flags structural non-identifiability; the encoding decides what
   is answerable before any mechanism is fitted.
2. **Falsification tests the graph, not the encoding.** A correct DAG over the wrong variables
   passes and still misleads.
3. **Check identification with two cheap diagnostics** before trusting any answer: Cramér's V
   between the cause columns (near 1 ⇒ the SCM cannot separate them, no matter how much data),
   and the connectivity of the comparison graph your identification strategy relies on (one
   `networkx` call).
4. **Identification can be earned upstream.** Domain structure (here: teammates share a car)
   can identify continuous latents that slot into the same graph; `gcm` composes cleanly with
   latents estimated by an external model.
5. **When roots are correlated, lead with `do()` and counterfactuals, in domain units.**
   Variance shares are graph- and population-dependent; interventional positions are not.

---

**Where everything comes from:** data is [f1db](https://github.com/f1db/f1db) (CC-BY-4.0);
the latents, their out-of-sample validation (80% held-out teammate head-to-heads; a pre-season
2026 forecast that scored 70% on the season's first half), and an interactive version of this
model live at [apex-attribution](https://github.com/bolt-chaos/apex-attribution) /
[the demo site](https://bolt-chaos.github.io/apex-attribution/). Proposed as a DoWhy example in
[py-why/dowhy#1752](https://github.com/py-why/dowhy/issues/1752).